# 00 — Setup

Validate Colab environment: Drive mount, repo clone, HF cache, symlinks.

**Runtime:** CPU is fine.

In [ ]:
GITHUB_REPO = 'https://github.com/evanpaul14/prompt-classifier.git'  # <-- update this
DRIVE_BASE  = '/content/drive/MyDrive/polygence'

import subprocess, os, sys

# GPU check (informational)
subprocess.run(['nvidia-smi'], check=False)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE = DRIVE_BASE

# Clone or pull repo
repo_dir = '/content/polygence'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', GITHUB_REPO, repo_dir], check=True)
else:
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)

os.chdir(repo_dir)
sys.path.insert(0, 'src')

# Install
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

# HF cache on Drive
os.environ['HF_DATASETS_CACHE'] = f'{DRIVE}/hf_cache/datasets'
os.environ['HF_HOME']           = f'{DRIVE}/hf_cache/hub'

# Symlinks
for subdir in ['data/processed', 'data/interim', 'models', 'reports']:
    drive_path = f'{DRIVE}/{subdir}'
    os.makedirs(drive_path, exist_ok=True)
    local_path = subdir
    if os.path.islink(local_path): os.remove(local_path)
    elif os.path.isdir(local_path) and not os.listdir(local_path): os.rmdir(local_path)
    if not os.path.exists(local_path): os.symlink(drive_path, local_path)

from prompt_classifier.seeds import set_all_seeds
set_all_seeds(42)
print('Setup complete!')

In [ ]:
# Verify symlinks and dirs
for d in ['data/processed', 'data/interim', 'models', 'reports']:
    print(d, '->', os.path.realpath(d) if os.path.islink(d) else '(real dir)')